In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "GPU is not enabled!"

In [ ]:
import os
import glob
import shutil

matches = glob.glob(
    "/kaggle/input/**/run_all_kaggle.sh",
    recursive=True
)

print("Found project:", matches)

assert matches, "Could not find run_all_kaggle.sh in /kaggle/input"

source_dir = os.path.dirname(matches[0])
project_dir = "/kaggle/working/AIR_Phase3_completed"

shutil.rmtree(project_dir, ignore_errors=True)
shutil.copytree(source_dir, project_dir)

os.chdir(project_dir)

print("Source:", source_dir)
print("Working directory:", os.getcwd())
print("\nProject files:")
print(os.listdir("."))

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    "import numpy, pandas, scipy; "
    "print('NumPy:', numpy.__version__); "
    "print('Pandas:', pandas.__version__); "
    "print('SciPy:', scipy.__version__); "
    "print('Scientific stack: OK')"
], check=True)

In [ ]:
!pip install -q --upgrade-strategy only-if-needed \
    "faiss-cpu<1.11" \
    "sentence-transformers[image]>=5.4" \
    "transformers>=4.57" \
    "qwen-vl-utils>=0.0.14" \
    "tabulate"

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    """
import torch
import torchvision
import numpy
import pandas
import scipy
import pyarrow
import transformers
import sentence_transformers
import faiss
import qwen_vl_utils

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS: OK")
print("Qwen VL Utils: OK")
print("\\nENVIRONMENT IS READY")
"""
], check=True)

In [ ]:
!python check_kaggle_gpu.py

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    """
import os
import pandas as pd

files = [
    "data/catalog_subset.parquet",
    "data/queries.jsonl",
    "data/qrels.jsonl",
    "run_all_kaggle.sh",
    "src/build_index.py",
    "src/retrieve.py",
    "src/rerank.py",
    "src/evaluate.py",
    "src/generate_answer.py",
    "src/tests.py",
]

for path in files:
    print(("OK     " if os.path.exists(path) else "MISSING"), path)

catalog = pd.read_parquet("data/catalog_subset.parquet")
print("\\nProducts:", len(catalog))
"""
], check=True)

In [ ]:
!DENSE_BATCH_SIZE=1 \
SPARSE_BATCH_SIZE=8 \
RERANK_BATCH_SIZE=16 \
bash run_all_kaggle.sh

In [ ]:
import os

expected = [
    "artifacts/embeddings/product_dense.npy",
    "artifacts/embeddings/product_ids.txt",
    "artifacts/embeddings/product_sparse.jsonl",
    "artifacts/index/dense_index.faiss",
    "artifacts/index/sparse_index.npz",
    "artifacts/index/index_manifest.json",
    "outputs/retrieval_results.jsonl",
    "outputs/reranked_results.jsonl",
    "data/qrels.jsonl",
    "reports/evaluation_metrics.csv",
    "reports/evaluation_report.md",
    "reports/error_analysis.md",
    "outputs/final_answers.jsonl",
]

all_ok = True

for path in expected:
    exists = os.path.exists(path)
    print("OK     " if exists else "MISSING", path)
    all_ok = all_ok and exists

print("\nAll final outputs available:", all_ok)

In [ ]:
import subprocess
import sys

for stage in range(6):
    print("\n" + "=" * 60)
    print(f"FINAL ACCEPTANCE TEST — STAGE {stage}")
    print("=" * 60)

    result = subprocess.run(
        [sys.executable, "src/tests.py", "--stage", str(stage)]
    )

    if result.returncode != 0:
        print(f"\n❌ Stage {stage} has failed checks.")
        break
else:
    print("\n✅ ALL STAGE 0–5 ACCEPTANCE TESTS PASSED")

In [ ]:
import shutil
import os

final_path = "/kaggle/working/AIR_Phase3_FINAL_COMPLETED"

shutil.make_archive(
    final_path,
    "zip",
    "/kaggle/working/AIR_Phase3_completed"
)

print("Final project created:")
print(final_path + ".zip")